# Datadog malicious software packages preview notebook

This notebook downloads the public **DataDog malicious-software-packages-dataset** repository, loads the per-ecosystem `manifest.json` files, and previews the first examples safely.

It **does not extract or execute** any malware samples.

**Source**  
The repository is public and currently states it contains **25,581 malicious software packages** across **npm, PyPI, IDE extensions, and AI Skills**. It says each ecosystem subdirectory contains a `manifest.json` file naming included packages and that sample archives are distributed as password-protected ZIP files. citeturn720793view1

In [ ]:
!pip -q install pandas requests

In [ ]:
import json
import ast
from pathlib import Path
from typing import Any
import pandas as pd
from IPython.display import display, Markdown

pd.set_option("display.max_colwidth", 200)

def truncate(text: Any, n: int = 300) -> str:
    if text is None:
        return ""
    s = str(text)
    return s if len(s) <= n else s[:n] + " …"

def maybe_parse_json(x: Any):
    if isinstance(x, (dict, list)):
        return x
    if not isinstance(x, str):
        return x
    x = x.strip()
    if not x:
        return x
    for fn in (json.loads, ast.literal_eval):
        try:
            return fn(x)
        except Exception:
            pass
    return x

def first_dialogue_excerpt(obj: Any, max_items: int = 4) -> str:
    parsed = maybe_parse_json(obj)
    if isinstance(parsed, list):
        parts = []
        for item in parsed[:max_items]:
            if isinstance(item, dict):
                role = item.get("role") or item.get("userType") or item.get("type") or "item"
                content = item.get("content") or item.get("text") or item.get("message") or item.get("action") or item.get("thought") or item
                parts.append(f"{role}: {truncate(content, 180)}")
            else:
                parts.append(truncate(item, 180))
        return "\n".join(parts)
    if isinstance(parsed, dict):
        return truncate(json.dumps(parsed, indent=2), 500)
    return truncate(parsed, 500)

def choose_first_existing(columns, candidates):
    for c in candidates:
        if c in columns:
            return c
    return None


import os
import requests
import zipfile
import tarfile
import io
from pathlib import Path

def download_file(url: str, dest: str, chunk_size: int = 2**20):
    dest = Path(dest)
    dest.parent.mkdir(parents=True, exist_ok=True)
    if dest.exists():
        print(f"Using cached file: {dest}")
        return dest
    print(f"Downloading {url} -> {dest}")
    with requests.get(url, stream=True, timeout=120) as r:
        r.raise_for_status()
        with open(dest, "wb") as f:
            for chunk in r.iter_content(chunk_size=chunk_size):
                if chunk:
                    f.write(chunk)
    return dest

def unzip_file(zip_path: str, out_dir: str):
    zip_path = Path(zip_path)
    out_dir = Path(out_dir)
    if out_dir.exists() and any(out_dir.iterdir()):
        print(f"Using cached extraction: {out_dir}")
        return out_dir
    out_dir.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path) as zf:
        zf.extractall(out_dir)
    return out_dir

In [ ]:
REPO_ZIP_URL = "https://github.com/DataDog/malicious-software-packages-dataset/archive/refs/heads/main.zip"
CACHE_DIR = Path("data_cache/datadog_malicious_packages")
ZIP_PATH = CACHE_DIR / "datadog_repo.zip"
EXTRACT_DIR = CACHE_DIR / "repo"

download_file(REPO_ZIP_URL, ZIP_PATH)
unzip_file(ZIP_PATH, EXTRACT_DIR)

roots = [p for p in EXTRACT_DIR.iterdir() if p.is_dir()]
root = roots[0] if len(roots) == 1 else EXTRACT_DIR
print("Repository root:", root)

In [ ]:
sample_root = root / "samples"
manifests = sorted(sample_root.glob("*/manifest.json"))
print("Found manifests:")
for m in manifests:
    print(" -", m)

def load_manifest(path: Path) -> pd.DataFrame:
    with open(path, "r", encoding="utf-8") as f:
        manifest = json.load(f)
    rows = []
    for package_name, affected_versions in manifest.items():
        rows.append({
            "ecosystem": path.parent.name,
            "package_name": package_name,
            "compromised_versions": None if affected_versions is None else ", ".join(map(str, affected_versions[:10])),
            "all_versions_malicious": affected_versions is None,
            "num_listed_versions": None if affected_versions is None else len(affected_versions),
        })
    return pd.DataFrame(rows)

manifest_df = pd.concat([load_manifest(p) for p in manifests], ignore_index=True)
display(manifest_df.head(20))

In [ ]:
counts = (
    manifest_df.groupby("ecosystem")
    .agg(
        packages=("package_name", "count"),
        all_versions_malicious=("all_versions_malicious", "sum"),
    )
    .reset_index()
    .sort_values("packages", ascending=False)
)
display(counts)

In [ ]:
zip_paths = sorted(sample_root.rglob("*.zip"))
zip_df = pd.DataFrame({
    "relative_path": [str(p.relative_to(root)) for p in zip_paths[:50]],
    "size_mb": [round(p.stat().st_size / 1e6, 4) for p in zip_paths[:50]],
})
display(zip_df.head(20))
print(f"Total sample archives discovered: {len(zip_paths)}")

In [ ]:
for ecosystem in sorted(manifest_df["ecosystem"].unique()):
    print(f"\n## {ecosystem}")
    display(manifest_df[manifest_df["ecosystem"] == ecosystem].head(10))

This preview notebook intentionally stays at the **metadata and package-list** level. Do not extract or run the sample archives outside a hardened malware-analysis environment.